In [ ]:
import yfinance as yf
import pandas as pd
from googlesearch import search
import requests
from bs4 import BeautifulSoup

#Get the historical data for the Microsoft index
index = "MSFT"
index_ticker = yf.Ticker(index)
data = index_ticker.history(period="max")
data = data.reset_index()
data['Date'] = data['Date'].dt.strftime('%Y-%m-%d')
data = data.drop(columns=['Dividends', 'Stock Splits'])
data['Date'] = pd.to_datetime(data['Date'])
data['Date'] = data['Date'].dt.strftime('%Y-%m-%d')
data 


,Date,Open,High,Low,Close,Volume
0,1986-03-13,0.054277,0.062259,0.054277,0.059598,1031788800
1,1986-03-14,0.059598,0.062791,0.059598,0.061726,308160000
2,1986-03-17,0.061726,0.063323,0.061726,0.062791,133171200
3,1986-03-18,0.062791,0.063323,0.060662,0.061194,67766400
4,1986-03-19,0.061194,0.061726,0.059598,0.060130,47894400
...,...,...,...,...,...,...
9923,2025-08-01,535.000000,535.799988,520.859985,524.109985,28977600
9924,2025-08-04,528.270020,538.250000,528.130005,535.640015,25349000
9925,2025-08-05,537.179993,537.299988,527.239990,527.750000,19171600
9926,2025-08-06,530.900024,531.700012,524.030029,524.940002,21355700


In [20]:
#Calculate the moving averages:
#Cumulative moving average:
window_size = 20
data['cumulative_MA'] = data['Close'].expanding().mean()

#Weighted average:
data['moving_average'] = data['Close'].rolling(window=window_size).mean()

#Exponentially weighted moving average:
data['exponentially_weighted_moving_average'] = data['Close'].ewm(span=window_size, adjust=False).mean()
data.dropna(inplace=True)
data

,Date,Open,High,Low,Close,Volume,cumulative_MA,moving_average,exponentially_weighted_moving_average
38,1986-05-07,0.067580,0.068112,0.066515,0.067580,5155200,0.065411,0.065411,0.065859
39,1986-05-08,0.067580,0.068112,0.067047,0.068112,3542400,0.065540,0.065810,0.066074
40,1986-05-09,0.068112,0.068112,0.067580,0.067580,6076800,0.065632,0.066130,0.066217
41,1986-05-12,0.067580,0.069708,0.067580,0.068112,10483200,0.065740,0.066449,0.066397
42,1986-05-13,0.068112,0.069176,0.068112,0.068644,3830400,0.065861,0.066795,0.066611
...,...,...,...,...,...,...,...,...,...
9923,2025-08-01,535.000000,535.799988,520.859985,524.109985,28977600,62.077709,509.028500,509.037236
9924,2025-08-04,528.270020,538.250000,528.130005,535.640015,25349000,62.125514,510.924501,511.570834
9925,2025-08-05,537.179993,537.299988,527.239990,527.750000,19171600,62.172514,512.481001,513.111707
9926,2025-08-06,530.900024,531.700012,524.030029,524.940002,21355700,62.219220,513.552501,514.238212


In [ ]:
def extract_info_from_url(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # Title of the page
        title = soup.title.string if soup.title else "No title"

        # Headline (usually in the first <h1>)
        headline_tag = soup.find("h1")
        headline = headline_tag.get_text(strip=True) if headline_tag else "No headline"

        # Meta description
        desc_tag = soup.find("meta", attrs={"name": "description"}) or \
                   soup.find("meta", attrs={"property": "og:description"})
        description = desc_tag["content"] if desc_tag and desc_tag.has_attr("content") else "No description"

        return {

            "title": title,
            "headline": headline,
            "description": description
        }

    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return None

def search_and_extract(company_name):
    query = [f"{company_name} site:cnn.com",f"{company_name} site:bbc.com"]  # Customize site if you want
    results = []

    for site in query:
        for url in search(site, num_results=5):  # Adjust number as needed
            info = extract_info_from_url(url)
            if info:
                results.append(info)

    for article in results:

        print("Title:", article["title"])
        print("Headline:", article["headline"])
        print("Description:", article["description"])
        print("-" * 80)

# Example usage
search_and_extract("Microsoft")


Title: MSFT Stock Quote Price and Forecast | CNN
Headline: Microsoft Corporation
Description: View Microsoft Corporation MSFT stock quote prices, financial information, real-time forecasts, and company news from CNN.
--------------------------------------------------------------------------------
Title: Microsoft has become the next $4 trillion company | CNN Business
Headline: Microsoft has become the next $4 trillion company
Description: Microsoft is set to soar past $4 trillion in market valuation for the first time on Thursday, as a blockbuster earnings report helps the tech behemoth become the second company after Nvidia to surpass the milestone.
--------------------------------------------------------------------------------
Title: Microsoft alerts businesses and governments to attacks on SharePoint servers | CNN Business
Headline: Microsoft alerts businesses and governments to attacks on SharePoint servers
Description: Microsoft has issued an alert about “active attacks” on serve

In [ ]:
#Design Linear regression model for the data
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

y = data['Close']
X = data.drop('Close', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lm = LinearRegression().fit(X_train, y_train)
X_transformed = lm.transform(X_test)

predictions = lm.predict(X_transformed)
accuracy = lm.score(y_test, predictions)



ValueError: could not convert string to float: '1999-07-08'